In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import spearmanr
from scipy.linalg import eig
from kneed import KneeLocator

# Reproducible project paths and output locations

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Run this notebook from inside the DELVE repository.")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))












from project_utils import configure_plots, ensure_output_dirs, save_figure
from functions import diffusion_map, LG_sym, calc_differential_vec

configure_plots()
ensure_output_dirs()


# Ensure imports work whether Jupyter runs from project root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Times New Roman'
plt.rcParams['mathtext.it'] = 'Times New Roman:italic'
plt.rcParams['mathtext.bf'] = 'Times New Roman:bold'

In [ ]:
def generate_simulation(
    n_samples=2000,
    rho=0.5,
    noise_std=0.02,
    random_state=42,
):
    """
    Algorithm 1 robustness simulation.

    theta is shared between the modalities, while psi_A1 and psi_B1
    contain both a shared component and a modality-specific residual:

        psi_A1 = rho * theta + sqrt(1-rho^2) * eta_A
        psi_B1 = rho * theta + sqrt(1-rho^2) * eta_B

    Returns
    -------
    XA, XB : observed data matrices
    latents : dictionary containing theta, psi_A1, psi_B1, eta_A, eta_B
    """
    if not 0 <= rho <= 1:
        raise ValueError("rho must be between 0 and 1.")

    rng = np.random.default_rng(random_state)

    theta = rng.normal(size=n_samples)
    eta_A = rng.normal(size=n_samples)
    eta_B = rng.normal(size=n_samples)

    psi_A1 = rho * theta + np.sqrt(1 - rho**2) * eta_A
    psi_B1 = rho * theta + np.sqrt(1 - rho**2) * eta_B

    XA = np.column_stack([theta, psi_A1])
    XB = np.column_stack([theta, psi_B1])

    XA += noise_std * rng.normal(size=XA.shape)
    XB += noise_std * rng.normal(size=XB.shape)

    latents = {
        "theta": theta,
        "psi_A1": psi_A1,
        "psi_B1": psi_B1,
        "eta_A": eta_A,
        "eta_B": eta_B,
    }

    return XA, XB, latents


if __name__ == "__main__":
    XA, XB, latents = generate_simulation(
        n_samples=1500,
        rho=0.6,
        noise_std=0.02,
        random_state=42,
    )

    print("XA shape:", XA.shape)
    print("XB shape:", XB.shape)
    print("corr(theta, psi_A1):",
          np.corrcoef(latents["theta"], latents["psi_A1"])[0, 1])
    print("corr(theta, psi_B1):",
          np.corrcoef(latents["theta"], latents["psi_B1"])[0, 1])

In [ ]:
# Core operators/eigenvectors for all methods on the representative sample
P1, Q1, K1 = diffusion_map(XA, adaptive=500)
P2, Q2, K2 = diffusion_map(XB, adaptive=500)

L1, d1, v1 = LG_sym(K1)
L2, d2, v2 = LG_sym(K2)

kl = KneeLocator(np.arange(len(d1)), d1, curve="convex", direction="decreasing",S=0.5)
tau1 = kl.knee
kl = KneeLocator(np.arange(len(d2)), d2, curve="convex", direction="decreasing",S=0.5)
tau2 = kl.knee
print(tau1)
print(tau2)

_, u1 = calc_differential_vec(L2, v1, tau1)
_, u2 = calc_differential_vec(L1, v2, tau2)

S = P2 @ Q1 + P1 @ Q2
D = P2 @ Q1 - P1 @ Q2

_, _ = eig(S)  # computed in original workflow; not used directly in final outputs
ea_vals, va = eig(D)
VA_imag = np.imag(va[:, np.argsort(np.imag(ea_vals))[::-1]])
VA_real = np.real(va[:, np.argsort(np.real(ea_vals))[::-1]])

g1 = np.diag(np.sum(K1, axis=1)) - K1
g2 = np.diag(np.sum(K2, axis=1)) - K2

m1 = g1 + 1e-6 * np.eye(g1.shape[0])
m2 = g2 + 1e-6 * np.eye(g2.shape[0])

fk1 = np.linalg.inv(m1 + m2) @ m1
fk2 = np.linalg.inv(m1 + m2) @ m2

fk_vals_1, eig_vec_fk_1 = eig(fk1)
fk_vals_2, eig_vec_fk_2 = eig(fk2)

eig_vec_fk_1 = eig_vec_fk_1[:, np.argsort(fk_vals_1)[::-1]]
eig_vec_fk_2 = eig_vec_fk_2[:, np.argsort(fk_vals_2)[::-1]]

In [ ]:
kl.plot_knee_normalized()

In [ ]:
corrs = {
    'DELVE': [
        abs(np.corrcoef(u2[:, 0], latents['eta_A'])[0, 1]),
    ],
    'Shnitzer et al. (real)': [
        abs(np.corrcoef(VA_real[:, 0], latents['eta_A'])[0, 1]),
    ],
    'Shnitzer et al. (imag)': [
        abs(np.corrcoef(VA_imag[:, 0], latents['eta_A'])[0, 1]),
    ],
    'FKT': [
        abs(np.corrcoef(eig_vec_fk_2[:, 0], latents['eta_A'])[0, 1]),
    ],
}

df_corr = pd.DataFrame(corrs, index=[r'$\eta$ ']).T

# Keep same exported orientation as original notebook
# df_corr.T.to_latex(TABLES_DIR / "Yoda_summary_table.tex", index=True, float_format='%.3f')
display(df_corr.T.round(3))

In [ ]:
rho_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

results_by_rho = {}

for rho in rho_values:

    print(f"\nRunning rho = {rho}")

    # ========================================================
    # Data generation
    # ========================================================

    XA, XB, latents = generate_simulation(
        n_samples=1500,
        rho=rho,
        noise_std=0.02,
        random_state=42,
    )

    print(
        "corr(theta, psi_A1):",
        np.corrcoef(
            latents["theta"],
            latents["psi_A1"],
        )[0, 1],
    )

    print(
        "corr(theta, psi_B1):",
        np.corrcoef(
            latents["theta"],
            latents["psi_B1"],
        )[0, 1],
    )

    # ========================================================
    # Diffusion operators
    # ========================================================

    P1, Q1, K1 = diffusion_map(XA, adaptive=500)
    P2, Q2, K2 = diffusion_map(XB, adaptive=500)

    L1, d1, v1 = LG_sym(K1)
    L2, d2, v2 = LG_sym(K2)

    # Number of eigenvalues required to explain 90% of the sum
    kl = KneeLocator(np.arange(len(d1)), d1, curve="convex", direction="decreasing")
    tau1 = kl.knee
    kl = KneeLocator(np.arange(len(d2)), d2, curve="convex", direction="decreasing")
    tau2 = kl.knee


    # ========================================================
    # DELVE differential vectors
    # ========================================================

    _, u1_all = calc_differential_vec(L2, v1, tau1)
    _, u2_all = calc_differential_vec(L1, v2, tau2)


    # ========================================================
    # Alternating-diffusion difference operator
    # ========================================================

    S = P2 @ Q1 + P1 @ Q2
    D = P2 @ Q1 - P1 @ Q2

    ea_vals, va = eig(D)

    imag_order = np.argsort(np.imag(ea_vals))[::-1]
    real_order = np.argsort(np.real(ea_vals))[::-1]

    VA_imag = np.imag(va[:, imag_order])
    VA_real = np.real(va[:, real_order])

    VA_imag_leading = VA_imag[:, 0]
    VA_real_leading = VA_real[:, 0]

    # ========================================================
    # FKT
    # ========================================================

    g1 = np.diag(np.sum(K1, axis=1)) - K1
    g2 = np.diag(np.sum(K2, axis=1)) - K2

    m1 = g1 + 1e-6 * np.eye(g1.shape[0])
    m2 = g2 + 1e-6 * np.eye(g2.shape[0])

    # solve(A, B) is numerically preferable to inv(A) @ B
    fk1 = np.linalg.solve(m1 + m2, m1)
    fk2 = np.linalg.solve(m1 + m2, m2)

    fk_vals_1, eig_vec_fk_1 = eig(fk1)
    fk_vals_2, eig_vec_fk_2 = eig(fk2)

    fk_order_1 = np.argsort(np.real(fk_vals_1))[::-1]
    fk_order_2 = np.argsort(np.real(fk_vals_2))[::-1]

    eig_vec_fk_1 = eig_vec_fk_1[:, fk_order_1]
    eig_vec_fk_2 = eig_vec_fk_2[:, fk_order_2]

    fkt_1_leading = np.real(eig_vec_fk_1[:, 0])
    fkt_2_leading = np.real(eig_vec_fk_2[:, 0])

    # ========================================================
    # Store the outputs
    # ========================================================

    results_by_rho[rho] = {
        "XA": XA,
        "XB": XB,
        "latents": latents,

        "tau1": tau1,
        "tau2": tau2,

        "u1": u1_all[:,0],
        "u2": u2_all[:,0],

        "fkt_1": fkt_1_leading,
        "fkt_2": fkt_2_leading,

        "VA_imag": VA_imag_leading,
        "VA_real": VA_real_leading,

    }

    

In [ ]:
rows = []

for rho, result in results_by_rho.items():

    theta = result["latents"]["theta"]
    psi_A1 = result["latents"]["psi_A1"]
    eta_A = result["latents"]["eta_A"]

    for method in ["u2", "fkt_2", "VA_imag", "VA_real"]:

        vector = np.real(result[method])

        rows.append({
            "rho": rho,
            "method": method,
            "corr_psi_A1": abs(spearmanr(vector, psi_A1).statistic),
            "corr_eta_A": abs(spearmanr(vector, eta_A).statistic),
            "corr_theta": abs(spearmanr(vector, theta).statistic),
        })

correlation_results = pd.DataFrame(rows)

In [ ]:
method_names = {
    "u2": "DELVE",
    "fkt_2": "FKT",
    "VA_real": "Shnitzer + (real)",
    "VA_imag": "Shnitzer + (imag)",
    
}

target_columns = {
    "corr_eta_A": r"$\psi_A$",
    # "corr_psi_A1": r"$X^A_{\cdot,1}$",
    "corr_theta": r"$\theta$",
}


fig, axes = plt.subplots(
    1,
    2,
    figsize=(8, 4),
    sharex=True,
    sharey=True,
)

for ax, (target_column, target_title) in zip(
    axes,
    target_columns.items(),
):
    for method, label in method_names.items():

        subset = (
            correlation_results[
                correlation_results["method"] == method
            ]
            .sort_values("rho")
        )

        ax.plot(
            subset["rho"],
            subset[target_column],
            marker="o",
            linewidth=2,
            markersize=6,
            label=label,
        )

    ax.set_title(target_title,fontsize=24)
    ax.set_xlabel(r" $\rho$",fontsize=22)
    ax.set_ylim(-0.02, 1.02)
    # ax.set_xlim(-0.02, 1.0)
    
    ax.grid(alpha=0.25)

axes[0].set_ylabel(r"Correlation [%]",fontsize=22)

handles, labels = axes[0].get_legend_handles_labels()


# fig.suptitle(
#     "Association of recovered vectors with shared and differential variables",
#     y=1.13,
# )

for ax in axes:
    ax.tick_params(axis='both', labelsize=16)

fig.legend(
    handles,
    labels,
    loc="upper center",
    ncol=4,
    bbox_to_anchor=(0.5, 1.05),
    fontsize=18,
)

plt.tight_layout(rect=[0, 0, 1, 0.90])

save_figure("robustness_simulation.pdf",
    bbox_inches="tight",
    pad_inches=0.1,
)

plt.show()